In [1]:
import os
import sys
from pathlib import Path
import json
current_dir = Path.cwd() 
project_root = os.path.dirname(os.path.dirname(current_dir))     # 项目根目录
sys.path.insert(0, project_root)
from config.config import SYSTEM_MESSAGE, INSTRUCTION

data_output_dir = os.path.join(project_root, "data")
train_data_path = os.path.join(data_output_dir, "train_data.json")
test_data_path = os.path.join(data_output_dir, "test_data.json")
data_path = os.path.join(data_output_dir, "output/lora_results_zero.json")
print(data_path,test_data_path)

/mnt/workspace/Graduate/ToxiCN_lora_rag/data/output/lora_results_zero.json /mnt/workspace/Graduate/ToxiCN_lora_rag/data/test_data.json


In [2]:
def trans_prompt(query):
    return INSTRUCTION + "\n以下是要分析的文本：" + query


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train = load_json(train_data_path)
test = load_json(test_data_path)
print(f"训练数据长度：{len(train)}")
print(f"训练数据长度：{len(test)}")


训练数据长度：6424
训练数据长度：1605


In [3]:
train_data = []
for d in train:
    prompt = trans_prompt(d['content'])
    train_data.append({
        'id': d['id'],
        'content':d['content'],
        "prompt": prompt,
        'output': d['output'],
    })

lora_train_path = os.path.join(project_root, "data", "lora_train_data.json")
with open(lora_train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)

In [4]:

test_data = []
for d in test:
    prompt = trans_prompt(d['content'])
    test_data.append({
        'id': d['id'],
        'content':d['content'],
        "prompt": prompt,
        'output': d['output'],
    })


lora_test_path = os.path.join(project_root, "data", "lora_test_data.json")
with open(lora_test_path, "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=4)

In [ ]:
from src.fine_Tuning.train import train_main
# #lora
# LORA_R = 8                
# LORA_ALPHA = 16           
# LORA_DROPOUT = 0.1         
# TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj"]  
# BIAS = "none"

# #训练参数
# NUM_TRAIN_EPOCHS = 5
# BATCH_SIZE = 4
# GRADIENT_ACCUMULATION_STEPS = 8
# LEARNING_RATE = 2e-5
# MAX_GRAD_NORM = 1.0
# SAVE_STEPS = 500
# LOGGING_STEPS = 50
# SEED = 42
# MAX_LENGTH = 512
# WEIGHT_DECAY = 0.01
# LR_SCHEDULER_TYPE = "cosine"      # 余弦退火
# EARLY_STOPPING_PATIENCE = 3
lora_save_path = "lora_zero_adapter"
train_main(lora_train_path, lora_save_path)

In [2]:
lora_test_path = os.path.join(project_root, "data", "lora_test_data.json")
lora_model_path = os.path.join(project_root, "output", "lora_zero_adapter/train")

In [3]:
from src.fine_Tuning.inference import inference_main

inference_main(lora_test_path, lora_model_path, data_path)

/root/miniconda3/envs/llmtoxi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


加载测试数据...
初始化推理引擎...
INFO 04-09 15:19:47 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'disable_log_stats': True, 'enable_lora': True, 'model': '/mnt/workspace/Graduate/ToxiCN_lora_rag/model/Qwen2.5-7B-Instruct'}
INFO 04-09 15:19:47 [model.py:533] Resolved architecture: Qwen2ForCausalLM
INFO 04-09 15:19:47 [model.py:1582] Using max model len 2048
INFO 04-09 15:19:47 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-09 15:19:47 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=11719) INFO 04-09 15:19:48 [core.py:103] Initializing a V1 LLM engine (v0.18.1) with config: model='/mnt/workspace/Graduate/ToxiCN_lora_rag/model/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='/mnt/workspace/Graduate/ToxiCN_lora_rag/model/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, dow

(EngineCore pid=11719) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=11719) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:21<01:03, 21.10s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:43<00:43, 21.76s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [01:05<00:21, 21.83s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:25<00:00, 21.09s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:25<00:00, 21.30s/it]
(EngineCore pid=11719) 


(EngineCore pid=11719) INFO 04-09 15:21:16 [default_loader.py:384] Loading weights took 85.27 seconds
(EngineCore pid=11719) INFO 04-09 15:21:16 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=11719) INFO 04-09 15:21:17 [gpu_model_runner.py:4566] Model loading took 14.32 GiB memory and 86.367860 seconds
(EngineCore pid=11719) INFO 04-09 15:21:22 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/79a954f42d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=11719) INFO 04-09 15:21:22 [backends.py:1048] Dynamo bytecode transform time: 4.97 s
(EngineCore pid=11719) INFO 04-09 15:21:25 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.376 s
(EngineCore pid=11719) INFO 04-09 15:21:25 [monitor.py:48] torch.compile took 7.79 s in total
(EngineCore pid=11719) INFO 04-09 15:21:25 [decorators.py:296] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_com

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 102/102 [00:10<00:00, 10.07it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 70/70 [00:05<00:00, 12.38it/s]


(EngineCore pid=11719) INFO 04-09 15:21:46 [gpu_model_runner.py:5746] Graph capturing finished in 17 secs, took 1.16 GiB
(EngineCore pid=11719) INFO 04-09 15:21:46 [gpu_worker.py:617] CUDA graph pool memory: 1.16 GiB (actual), 0.92 GiB (estimated), difference: 0.24 GiB (20.5%).
(EngineCore pid=11719) INFO 04-09 15:21:46 [core.py:281] init engine (profile, create kv cache, warmup model) took 29.00 seconds
INFO 04-09 15:21:46 [llm.py:391] Supported tasks: ['generate']
提取所有问题...
批量推理...


批量推理:   0%|          | 0/201 [00:00<?, ?it/s]

WARNING 04-09 15:21:47 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


批量推理: 100%|██████████| 201/201 [05:30<00:00,  1.64s/it]


(EngineCore pid=11719) INFO 04-09 15:27:17 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=11719) INFO 04-09 15:27:17 [core.py:1224] Shutdown complete
保存结果...
结果保存至 /mnt/workspace/Graduate/ToxiCN_lora_rag/data/output/lora_results_zero.json


In [4]:
from src.eval import main
main(data_path)

四元组硬匹配结果： (0.17925531914893617, 0.17718191377497372, 0.17821258593336858)
四元组软匹配结果： (0.3595744680851064, 0.35541535226077814, 0.3574828133262824)
三元组硬匹配结果： (0.2154255319148936, 0.21293375394321767, 0.21417239555790588)
三元组软匹配结果： (0.44627659574468087, 0.4411146161934805, 0.44368059227921736)
二元组硬匹配结果： (0.27180851063829786, 0.268664563617245, 0.27022739291380227)
二元组软匹配结果： (0.5574468085106383, 0.5509989484752892, 0.5542041248016923)
单元素硬匹配结果： (0.33351063829787236, 0.32965299684542587, 0.3315705975674247)
单元素软匹配结果： (0.6728723404255319, 0.6650893796004206, 0.6689582231623479)


{'quadruple_hard_match': (0.17925531914893617,
  0.17718191377497372,
  0.17821258593336858),
 'quadruple_soft_match': (0.3595744680851064,
  0.35541535226077814,
  0.3574828133262824),
 'triple_hard_match': (0.2154255319148936,
  0.21293375394321767,
  0.21417239555790588),
 'triple_soft_match': (0.44627659574468087,
  0.4411146161934805,
  0.44368059227921736),
 'pair_hard_match': (0.27180851063829786,
  0.268664563617245,
  0.27022739291380227),
 'pair_soft_match': (0.5574468085106383,
  0.5509989484752892,
  0.5542041248016923),
 'single_element_hard_match': (0.33351063829787236,
  0.32965299684542587,
  0.3315705975674247),
 'single_element_soft_match': (0.6728723404255319,
  0.6650893796004206,
  0.6689582231623479)}